# Stage A — keyword optimization & error analysis (on SAL full papers)

Stage A is the localizer: it slices each full paper down to the regions likely to hold gaps, so Stage B
only classifies a few % of sentences. Here we **measure and optimize it on labeled full papers** — the
**SAL/RCT** dataset (200 full articles, every sentence labeled limitation/not), the only public set that
gives a whole paper with exhaustive per-sentence labels, so we can score Stage-A **recall** (did the slice
keep the real limitation sentences?) and **drop rate** (how much it threw away).

We also create a **canonical train/val/test split by paper** and save it so **Stage B uses the same split** —
A and B are then trained/tuned/evaluated on identical papers.

> SAL is biomedical / limitation-only. FWS (future-work) can't be used here — it is pre-sliced to ~7
> sentences/paper, so there is no full paper to localize over.

## Setup (imports Stage A, loads SAL)

In [ ]:
import json, re, sys
from pathlib import Path
import numpy as np, pandas as pd
from collections import Counter

ROOT=Path.cwd()
while not (ROOT/"data"/"external").exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
# Stage A is imported (frozen, solved step); we optimize its cue lists around it
from gap2idea.pipeline.gap_funnel import slice_with_midpaper_anchors, token_containment, _LIMIT_CUES

sal=pd.read_csv(ROOT/"data/external/sal_rct/train.csv")
sal["pos"]=sal["limitation_annotation"].astype(str).str.contains("True")
print("SAL:",len(sal),"sentences /",sal.pmid.nunique(),"full papers | limitation prevalence",
      f"{sal.pos.mean()*100:.1f}%")

## Shared train/val/test split (used by Stage A **and** Stage B)

In [ ]:
# Canonical paper-disjoint split (70/15/15 by pmid) — SHARED by Stage A and Stage B.
SP=ROOT/"data/external/sal_splits.json"
if SP.exists():
    split=json.load(open(SP)); print("loaded shared split", {k:len(v) for k,v in split.items()})
else:
    pm=sorted(sal.pmid.unique()); rng=np.random.RandomState(0); rng.shuffle(pm); n=len(pm)
    split={"train":pm[:int(n*.7)],"val":pm[int(n*.7):int(n*.85)],"test":pm[int(n*.85):]}
    json.dump(split,open(SP,"w")); print("created shared split", {k:len(v) for k,v in split.items()})
TR,VA,TE=set(split["train"]),set(split["val"]),set(split["test"])
# NOTE: Stage B can load the same file so A & B train/eval on identical papers:
#   split = json.load(open("data/external/sal_splits.json"))

## Stage-A per split (illustrative — unstable at 18 papers)

In [ ]:
def stage_a(pmset):
    """Run Stage A over the full papers in pmset; return recall of limitation positives, drop, misses."""
    hit=tot=sl=ft=0; missed=[]
    for pid,g in sal.groupby("pmid"):
        if pid not in pmset: continue
        sents=[str(s) for s in g["sentences"]]; full=" ".join(sents)
        regs=slice_with_midpaper_anchors(full); st=" ".join(s for r in regs for s in r.sentences)
        sl+=sum(len(r.sentences) for r in regs); ft+=len(sents)
        for _,row in g[g.pos].iterrows():
            tot+=1
            if token_containment(str(row["sentences"]),st)>=0.7: hit+=1
            else: missed.append((re.sub(r"\|.*","",str(row.get("section_name",""))), str(row["sentences"])))
    return {"recall":round(hit/tot,3),"hit":hit,"tot":tot,"drop_%":round(100*(1-sl/ft),1),"missed":missed}

rows=[]
for nm,s in [("train",TR),("val",VA),("test",TE)]:
    r=stage_a(s); rows.append({"split":nm,"recall":r["recall"],"drop_%":r["drop_%"],"positives":r["tot"]})
pd.DataFrame(rows).set_index("split")

Single-split numbers swing a lot: train 0.861 · val 0.766 · test 0.935. That spread is **small-sample
noise** — a few hard papers landing in val drag it down. **Don't trust a single split; use cross-validation
below.**

## Stage-A — 5-fold paper-grouped cross-validation (the stable estimate)
Average over 5 folds (all 120 papers held out once). This is the number to report.

In [ ]:
# 5-fold paper-grouped CV: baseline + cue-augmented Stage A
from sklearn.model_selection import GroupKFold
NEW=re.compile(r"\b(a limitation|limitation of|we cannot|we could not|it is possible that|"
               r"may have been|may not be|may not have|is possible that)\b", re.I)
WIN=2
# precompute per-paper slice, augmented slice, and positives once
pdata={}
for pid,g in sal.groupby("pmid"):
    sents=[str(s) for s in g["sentences"]]; full=" ".join(sents)
    regs=slice_with_midpaper_anchors(full); st=" ".join(s for r in regs for s in r.sentences)
    keep=set()
    for i,s in enumerate(sents):
        if NEW.search(s):
            for j in range(max(0,i-WIN),min(len(sents),i+WIN+1)): keep.add(j)
    pdata[pid]=dict(nsent=len(sents), nslice=sum(len(r.sentences) for r in regs), nkeep=len(keep),
                    st=st, aug=st+" "+" ".join(sents[j] for j in keep),
                    pos=[str(x) for x in g[g.pos]["sentences"]])
pm=list(pdata); baseR=[]; augR=[]; baseD=[]; augD=[]
for _,te in GroupKFold(n_splits=5).split(pm,[0]*len(pm),groups=pm):
    tp=[pm[i] for i in te]; bh=ah=tot=bsl=asl=ft=0
    for pid in tp:
        d=pdata[pid]; ft+=d["nsent"]; bsl+=d["nslice"]; asl+=d["nslice"]+d["nkeep"]
        for s in d["pos"]:
            tot+=1
            if token_containment(s,d["st"])>=0.7: bh+=1
            if token_containment(s,d["aug"])>=0.7: ah+=1
    baseR.append(bh/tot); augR.append(ah/tot); baseD.append(100*(1-bsl/ft)); augD.append(100*(1-asl/ft))
import numpy as _np
print(f"baseline recall {_np.mean(baseR):.3f} ± {_np.std(baseR):.3f} | drop {_np.mean(baseD):.1f}%")
print(f"+cues    recall {_np.mean(augR):.3f} ± {_np.std(augR):.3f} | drop {_np.mean(augD):.1f}%")
print("per-fold baseline:",[round(x,3) for x in baseR])
print("per-fold +cues   :",[round(x,3) for x in augR])

**Reference (5-fold CV):** baseline recall **0.861 ± 0.054** (drop 82.2%) → **+cues 0.880 ± 0.047**
(drop 79.5%). Per-fold baseline `[0.873, 0.806, 0.849, 0.816, 0.958]` — the ±0.05 spread is why single
splits mislead. The +1.9pp cue gain is below the std but **consistent in every fold** (never negative), so
it's a real small effect.

## Where does Phase A make mistakes? (pooled over all papers)

In [ ]:
# Error analysis: WHERE does Phase A miss? (pooled over ALL papers)
res=stage_a(set(sal.pmid.unique())); miss=res["missed"]
print(f"missed positives: {len(miss)}")
print("by section:", Counter(s for s,_ in miss).most_common())
cue=sum(1 for _,x in miss if _LIMIT_CUES.search(x))
print(f"contain a current keyword cue: {cue}/{len(miss)}  | CUE-LESS: {len(miss)-cue}/{len(miss)}")
print("\nexamples of missed (cue-less, mid-paper Discussion):")
shown=0
for sec,x in miss:
    if not _LIMIT_CUES.search(x) and shown<6:
        print(f"  [{sec}] {x[:100]}"); shown+=1

## Minimal summary — Phase A's mistakes, in simple words

- Stage A keeps ~82% of the paper out and still finds **86% ± 5% (5-fold CV)** of the real limitation sentences.
- **Every missed limitation is in the Discussion section** — and **~96% of the misses have no limitation
  keyword at all** (they're "cue-less").
- The missed ones are plain-English limitations with no trigger word, e.g.
  *"This study was not blinded…"*, *"…the determination was subjective…"*,
  *"A lower dose might have produced a different outcome…"*, *"Our patients presented relatively late…"*.
- **Why keywords can't fix most of it:** there is no word to match — these read like normal sentences.
  Adding cues recovers only a couple; the rest need either (a) keeping the **whole Discussion** (more recall,
  more drop) or (b) a **meaning-based** check (the embedding head) at Stage A.

**One line:** Stage A misses *cue-less, mid-Discussion* limitations — keyword tuning gives only a small bump
(+2pp); the real fix is **structural**: keeping the whole Discussion/Limitation section lifts recall from
**0.861 → 0.988** (Path A, below), which is what a recall-first stage should do.

## Path A — section-aware keep (near-100% recall)
Stage A is recall-first: anything it drops is lost. The misses are mid-Discussion cue-less limitations, so
instead of the positional gate we **keep the whole Discussion/Conclusion/Limitation section**. (Here we use
SAL's section labels = a "perfect heading detection" upper bound; on real PDFs this needs GROBID or good
heading detection — see note.)

In [ ]:
# Path A: keep the WHOLE Discussion/Conclusion/Limitation section (section-aware), 5-fold CV
from sklearn.model_selection import GroupKFold
SEC=re.compile(r"discussion|conclusion|limitation|weakness|strength|future|outlook", re.I)
pa={}
for pid,g in sal.groupby("pmid"):
    g=g.reset_index(drop=True); sents=[str(s) for s in g["sentences"]]; secs=[str(x) for x in g["section_name"]]
    keep=[i for i,sc in enumerate(secs) if SEC.search(sc)]
    pa[pid]=dict(nsent=len(sents), nkeep=len(keep),
                 text=" ".join(sents[i] for i in keep), pos=[str(x) for x in g[g.pos]["sentences"]])
pmA=list(pa); R=[]; D=[]
for _,te in GroupKFold(5).split(pmA,[0]*len(pmA),groups=pmA):
    tp=[pmA[i] for i in te]; hit=tot=sl=ft=0
    for pid in tp:
        d=pa[pid]; ft+=d["nsent"]; sl+=d["nkeep"]
        for s in d["pos"]:
            tot+=1
            if token_containment(s,d["text"])>=0.7: hit+=1
    R.append(hit/tot); D.append(100*(1-sl/ft))
import numpy as _np
print(f"Path A (whole target sections): recall {_np.mean(R):.3f} ± {_np.std(R):.3f} | drop {_np.mean(D):.1f}%")
print("per-fold:",[round(x,3) for x in R])

**Result (5-fold CV):**
| Stage A | recall | drop |
|---|---|---|
| current (positional) | 0.861 ± 0.054 | 82.2% |
| **A: whole target sections** | **0.988 ± 0.020** | 75.8% |
| A + cues | 0.990 ± 0.020 | 75.4% |

Near-100% recall (4/5 folds = 1.000) and **more stable** (std halved), for ~6pp more kept. This is the right
tradeoff for a recall-first stage.

**Implementation / caveat:** this used SAL's `section_name` (perfect section detection). To get it on real
arXiv PDFs, Stage A must detect the Discussion/Limitations sections from the text — reliable via **GROBID**
(`slice_grobid_regions` already keeps whole classified sections), or an improved heading detector on the
PyMuPDF text. The realized recall then depends on section-detection quality, not on this ceiling.

## Data-driven cue mining (train only, to avoid leakage)

In [ ]:
# Mine high-lift limitation cues from TRAIN positives (data-driven, paper-disjoint)
def toks(s): return re.findall(r"[a-z]+", s.lower())
def ngrams(ws,n): return [" ".join(ws[i:i+n]) for i in range(len(ws)-n+1)]
trdf=sal[sal.pmid.isin(TR)]; base=trdf.pos.mean()
pos_c,all_c=Counter(),Counter()
for _,r in trdf.iterrows():
    grams=set(sum((ngrams(toks(str(r["sentences"])),n) for n in (1,2,3)),[]))
    for g in grams:
        all_c[g]+=1
        if r["pos"]: pos_c[g]+=1
cand=[(g,round(pos_c[g]/a,3),pos_c[g],a,round((pos_c[g]/a)/base,1))
      for g,a in all_c.items() if a>=15 and pos_c[g]>=6 and pos_c[g]/a>=0.25]
cand.sort(key=lambda x:-x[4])
print(f"base positive rate {base:.3f}. Top mined cues (phrase, precision, pos, total, lift):")
for c in cand[:15]: print("  ",c)

## Apply curated transferable cues → TEST recall/drop (before vs after)

In [ ]:
# Apply curated TRANSFERABLE cues (exclude biomed-specific) and measure TEST recall/drop
NEW=re.compile(r"\b(a limitation|limitation of|we cannot|we could not|it is possible that|"
               r"may have been|may not be|may not have|is possible that)\b", re.I)
WIN=2
def stage_a_aug(pmset):
    hit=tot=sl=ft=0
    for pid,g in sal.groupby("pmid"):
        if pid not in pmset: continue
        g=g.reset_index(drop=True); sents=[str(s) for s in g["sentences"]]; full=" ".join(sents)
        regs=slice_with_midpaper_anchors(full); st=" ".join(s for r in regs for s in r.sentences)
        keep=set()
        for i,s in enumerate(sents):
            if NEW.search(s):
                for j in range(max(0,i-WIN),min(len(sents),i+WIN+1)): keep.add(j)
        aug=st+" "+" ".join(sents[j] for j in keep)
        sl+=sum(len(r.sentences) for r in regs)+len(keep); ft+=len(sents)
        for _,row in g[g.pos].iterrows():
            tot+=1
            if token_containment(str(row["sentences"]),aug)>=0.7: hit+=1
    return round(hit/tot,3), round(100*(1-sl/ft),1)
base=stage_a(TE); ar,ad=stage_a_aug(TE)
print(f"TEST baseline : recall {base['recall']} | drop {base['drop_%']}%")
print(f"TEST +cues    : recall {ar} | drop {ad}%")

**Reference (canonical split, cues mined on train):** recall before→after — train 0.861→0.882,
val 0.766→0.798, test 0.935→0.946; drop ~82%→~80% on each. Small but real and consistent — confirms the
ceiling: keyword tuning helps ~+2pp; most missed gaps are cue-less.

### Transferable cues worth adding to the arXiv pipeline
`a limitation`, `limitation of`, `we cannot`, `we could not`, `it is possible that`, `may have been`,
`may not be`, `may not have`, `is possible that`.
(Biomed-specific ones like `selection bias`, `the small sample` are **not** ported — they won't transfer to CS.)

### How Stage B reuses this
`data/external/sal_splits.json` holds the paper-disjoint train/val/test. Stage B loads the same file for
its SAL portion so Phase A and Phase B are trained/tuned/evaluated on identical papers — no leakage between
the stages.